# Stock Price Target Prediction — ML Analytics Platform

**Version 3.0.0** — Unified ETL Pipeline with Phase 9.1-9.8 Workflow
**Model Version: v9_9**

## Business Objective

**Primary Goal**: Predict Stock Price Targets for all stocks in the portfolio to support
investment decisions and portfolio optimization.

**Target Variable**: `price_target` for regression modeling

## Quick Reference Navigation
- [Section 0](#section-0-configuration-and-setup): Configuration and Setup
- [Section 1](#section-1-unified-etl-pipeline): Phase 9.1-9.3 Unified ETL Pipeline
- [Section 2](#section-2-feature-selection): Phase 9.3 Feature Selection
- [Section 3](#section-3-classification): Phase 9.4 Multi-Class Event Classification
- [Section 4](#section-4-regression): Phase 9.5 Sector-Optimized Regression
- [Section 5](#section-5-evaluation): Phase 9.6 Model Evaluation
- [Section 6](#section-6-analytics): Phase 9.7 Stock Ranking Analytics
- [Section 7](#section-7-reporting): Phase 9.8 Comprehensive Reporting

## Workflow Overview

| Section | Phase | Description | Key Outputs |
|---------|-------|-------------|-------------|
| 0 | Setup | Environment, paths, logging, seed | Config validated |
| 1 | 9.1-9.3 | Unified ETL + Feature Engineering | `all_stocks_preprocessed` |
| 2 | 9.3 | Feature Selection | `all_stocks_selected` |
| 3 | 9.4 | Event Classification | `all_stocks_classification` |
| 4 | 9.5 | Sector-Optimized Regression | Trained models, predictions |
| 5 | 9.6 | Evaluation and Error Analysis | Metrics by sector |
| 6 | 9.7 | Analytics: Rankings, Mispricing | Rankings DataFrame |
| 7 | 9.8 | Reporting: Artifacts, Dashboards | Output files |

In [ ]:
# =============================================================================
# Section 0: Configuration and Setup
# =============================================================================

import os
import warnings
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# Logging Configuration
# -----------------------------------------------------------------------------
logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
        )
logger = logging.getLogger(__name__)

# -----------------------------------------------------------------------------
# Target Configuration (code_guidelines.md Section 2.2)
# -----------------------------------------------------------------------------
TARGET_COL = 'price_target'
TARGET_COL_FALLBACK = 'last_price'

# -----------------------------------------------------------------------------
# Data Split Configuration
# -----------------------------------------------------------------------------
TEST_SIZE = 0.2
TRAIN_SIZE = 1 - TEST_SIZE
CV_FOLDS = 5

# -----------------------------------------------------------------------------
# Quantile Regression Configuration
# -----------------------------------------------------------------------------
QUANTILES = [0.1, 0.5, 0.9]
LOWER_QUANTILE = QUANTILES[0]
MEDIAN_QUANTILE = QUANTILES[1]
UPPER_QUANTILE = QUANTILES[2]

# -----------------------------------------------------------------------------
# Sector Analysis Configuration
# -----------------------------------------------------------------------------
MIN_SECTOR_SAMPLES = 20
MAX_SECTOR_WEIGHT = 0.25
MAX_SINGLE_POSITION = 0.10

# -----------------------------------------------------------------------------
# Outlier Detection Configuration
# -----------------------------------------------------------------------------
IQR_MULTIPLIER = 1.5
ZSCORE_THRESHOLD = 3.0
WINSORIZE_LOWER = 0.01
WINSORIZE_UPPER = 0.99

# -----------------------------------------------------------------------------
# Confidence Thresholds
# -----------------------------------------------------------------------------
CONFIDENCE_LEVEL = 0.80
ALPHA = 1 - CONFIDENCE_LEVEL

# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------
RANDOM_SEED = int(os.getenv('RANDOM_SEED', '42'))
MODEL_VERSION = os.getenv('MODEL_VERSION', 'v9_9')
np.random.seed(RANDOM_SEED)

# -----------------------------------------------------------------------------
# Directory Configuration
# -----------------------------------------------------------------------------
DATA_DIR = Path(os.getenv('DATA_DIR', 'data'))
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'outputs'))
MODEL_DIR = Path(os.getenv('MODEL_DIR', 'models'))

# Create output subdirectories
OUTPUT_SUBDIRS = [
    'eda', 'preprocessing', 'features', 'classification',
    'regression', 'evaluation', 'analytics', 'reporting',
    'plots', 'governance'
    ]
for subdir in OUTPUT_SUBDIRS:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

logger.info(f"Output directory: {OUTPUT_DIR}")
logger.info(f"Model version: {MODEL_VERSION}")

In [ ]:
def validate_configuration():
    """
    Validate all configuration constants meet required constraints.

    Raises:
        ValueError: If any configuration constant is invalid
    """
    # Validate target columns
    if not TARGET_COL or not isinstance(TARGET_COL, str):
        raise ValueError(f"TARGET_COL must be a non-empty string, got: {TARGET_COL}")
    if not TARGET_COL_FALLBACK or not isinstance(TARGET_COL_FALLBACK, str):
        raise ValueError(f"TARGET_COL_FALLBACK must be a non-empty string, got: {TARGET_COL_FALLBACK}")

    # Validate split configuration
    if not (0 < TEST_SIZE < 1):
        raise ValueError(f"TEST_SIZE must be between 0 and 1, got: {TEST_SIZE}")
    if not (0 < TRAIN_SIZE < 1):
        raise ValueError(f"TRAIN_SIZE must be between 0 and 1, got: {TRAIN_SIZE}")
    if not abs((TRAIN_SIZE + TEST_SIZE) - 1.0) < 0.01:
        raise ValueError(f"TRAIN_SIZE + TEST_SIZE must equal 1.0, got: {TRAIN_SIZE + TEST_SIZE}")

    # Validate CV folds
    if not isinstance(CV_FOLDS, int) or CV_FOLDS < 2:
        raise ValueError(f"CV_FOLDS must be an integer >= 2, got: {CV_FOLDS}")

    # Validate quantiles
    if not QUANTILES or not isinstance(QUANTILES, list):
        raise ValueError(f"QUANTILES must be a non-empty list, got: {QUANTILES}")
    for q in QUANTILES:
        if not (0 <= q <= 1):
            raise ValueError(f"All quantiles must be between 0 and 1, got: {q}")
    if len(QUANTILES) != len(set(QUANTILES)):
        raise ValueError(f"QUANTILES must not contain duplicates, got: {QUANTILES}")
    if QUANTILES != sorted(QUANTILES):
        raise ValueError(f"QUANTILES must be monotonically increasing, got: {QUANTILES}")

    # Validate sector configuration
    if not isinstance(MIN_SECTOR_SAMPLES, int) or MIN_SECTOR_SAMPLES < 1:
        raise ValueError(f"MIN_SECTOR_SAMPLES must be a positive integer, got: {MIN_SECTOR_SAMPLES}")
    if not (0 < MAX_SECTOR_WEIGHT <= 1):
        raise ValueError(f"MAX_SECTOR_WEIGHT must be between 0 and 1, got: {MAX_SECTOR_WEIGHT}")
    if not (0 < MAX_SINGLE_POSITION <= 1):
        raise ValueError(f"MAX_SINGLE_POSITION must be between 0 and 1, got: {MAX_SINGLE_POSITION}")

    # Validate outlier detection
    if IQR_MULTIPLIER <= 0:
        raise ValueError(f"IQR_MULTIPLIER must be positive, got: {IQR_MULTIPLIER}")
    if ZSCORE_THRESHOLD <= 0:
        raise ValueError(f"ZSCORE_THRESHOLD must be positive, got: {ZSCORE_THRESHOLD}")
    if not (0 <= WINSORIZE_LOWER < 0.5):
        raise ValueError(f"WINSORIZE_LOWER must be between 0 and 0.5, got: {WINSORIZE_LOWER}")
    if not (0.5 < WINSORIZE_UPPER <= 1):
        raise ValueError(f"WINSORIZE_UPPER must be between 0.5 and 1, got: {WINSORIZE_UPPER}")

    # Validate confidence configuration
    if not (0 < CONFIDENCE_LEVEL < 1):
        raise ValueError(f"CONFIDENCE_LEVEL must be between 0 and 1, got: {CONFIDENCE_LEVEL}")
    if not abs(ALPHA - (1 - CONFIDENCE_LEVEL)) < 0.01:
        raise ValueError(f"ALPHA must equal (1 - CONFIDENCE_LEVEL), got: {ALPHA}")

    print("✓ All configuration constants validated successfully")


# Run validation immediately
validate_configuration()